# SASRec Stage 3 Refine Attention-Bias Multi-Task BPI2012 Colab Train 08

This notebook extends the Stage 3 attention-bias multi-task experiments to the
`refine_ml50_do035` backbone.

Main comparison groups:
- `refine_single_task`
- `refine_attnbias_single_task`
- `refine_multi_task_w1.0`
- `refine_attnbias_multi_task_w1.0`
- `refine_attnbias_multi_task_w0.1`

Main comparison metric:
- `full ranking + NDCG@10`


In [1]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


torch version: 2.11.0+cu128
cuda available: True
gpu name: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only_stage3_v2'
BASELINE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10'
SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10'
MULTITASK_BASELINE_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10_v2'
MULTITASK_REFINE_ATTNBIAS_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_refine_attention_bias_multitask_ndcg10_v2'
MULTITASK_REFINE_ATTNBIAS_W01_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_refine_attention_bias_multitask_w01_ndcg10_v2'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DATA_DIR:', DATA_DIR)
print('BASELINE_NDCG10_OUTPUT_DIR:', BASELINE_NDCG10_OUTPUT_DIR)
print('SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR:', SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR)
print('MULTITASK_BASELINE_OUTPUT_DIR:', MULTITASK_BASELINE_OUTPUT_DIR)
print('MULTITASK_REFINE_ATTNBIAS_OUTPUT_DIR:', MULTITASK_REFINE_ATTNBIAS_OUTPUT_DIR)
print('MULTITASK_REFINE_ATTNBIAS_W01_OUTPUT_DIR:', MULTITASK_REFINE_ATTNBIAS_W01_OUTPUT_DIR)
print('NOTEBOOK_DIR:', NOTEBOOK_DIR)


DATA_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2
BASELINE_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10
SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10
MULTITASK_BASELINE_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10_v2
MULTITASK_REFINE_ATTNBIAS_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_refine_attention_bias_multitask_ndcg10_v2
MULTITASK_REFINE_ATTNBIAS_W01_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_refine_attention_bias_multitask_w01_ndcg10_v2
NOTEBOOK_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/notebooks


In [4]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$BASELINE_NDCG10_OUTPUT_DIR"
!mkdir -p "$SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR"
!mkdir -p "$MULTITASK_BASELINE_OUTPUT_DIR"
!mkdir -p "$MULTITASK_REFINE_ATTNBIAS_OUTPUT_DIR"
!mkdir -p "$MULTITASK_REFINE_ATTNBIAS_W01_OUTPUT_DIR"


In [5]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction
!git pull


/content
/content/time-aware-behavior-prediction
Already up to date.


In [6]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


/content/time-aware-behavior-prediction
created requirements_colab.txt


In [7]:
!pip install -r requirements_colab.txt


## Prepare Stage 3 processed dataset

This notebook regenerates the Stage 3 dataset into the versioned Drive folder
before training, so the run does not depend on any stale local processed files.


In [8]:
%cd /content/time-aware-behavior-prediction
!python scripts/regenerate_stage3_processed_dataset.py --output-dir "$DATA_DIR" --backup-existing
!ls "$DATA_DIR"


/content/time-aware-behavior-prediction
[info] moved existing output to backup: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2__backup_20260607_113944
[ok] regenerated Stage 3 processed dataset at: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2
[ok] metadata written to: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2/stage3_dataset_metadata.json
{
  "timestamp": 0,
  "delta_prev_seconds": 0,
  "delta_start_seconds": 0,
  "delta_next_seconds": 0
}
events_complete_only_filtered.csv  sasrec_interactions.csv	 user_map.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   stage3_dataset_metadata.json


In [9]:
%cd /content/time-aware-behavior-prediction
!mkdir -p data/processed
!rm -rf data/processed/bpi2012_complete_only_stage3_v2
!cp -r "$DATA_DIR" data/processed/
!ls data/processed/bpi2012_complete_only_stage3_v2


/content/time-aware-behavior-prediction
events_complete_only_filtered.csv  sasrec_interactions.csv	 user_map.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   stage3_dataset_metadata.json


In [10]:
import pandas as pd

time_features_path = 'data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv'
df = pd.read_csv(time_features_path)
required_cols = ['delta_prev_seconds', 'delta_start_seconds', 'delta_next_seconds']
missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f'Missing required Stage 3 columns: {missing}')

print('Stage 3 processed file is ready.')
print(df.columns.tolist())
df[['user_id', 'event_idx', 'delta_prev_seconds', 'delta_start_seconds', 'delta_next_seconds']].head()


Stage 3 processed file is ready.
['case_id', 'activity', 'lifecycle', 'timestamp', 'event_idx', 'delta_prev_seconds', 'delta_start_seconds', 'delta_next_seconds', 'user_id', 'item_id']


,user_id,event_idx,delta_prev_seconds,delta_start_seconds,delta_next_seconds
0,1,0,0.000,0.000,0.334
1,1,1,0.334,0.334,53.026
2,1,2,53.026,53.360,39785.402
3,1,3,39785.402,39838.762,145.935
4,1,4,145.935,39984.697,-0.000


## Experiment design

This notebook tests the `refine_ml50_do035` backbone with:
- single-task attention bias (`Stage 2` reference)
- multi-task attention bias with `time_loss_weight=1.0`
- multi-task attention bias with `time_loss_weight=0.1`

Fixed settings:
- backbone: `refine_ml50_do035`
- time-aware backbone: `delta_start + 9-bucket attention bias`
- multi-task outputs: `next activity + next time`
- time target: `delta_next_seconds`
- time target transform: `log1p`
- time loss: `huber`
- best epoch criterion: `full_valid_ndcg@10`
- final comparison uses all 3 seeds: `42`, `2024`, `7`


## Check prerequisite reference runs


In [11]:
from pathlib import Path

baseline_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]
single_task_attnbias_runs = [
    'attnbias_dstart_ml50_do035_b9_s42',
    'attnbias_dstart_ml50_do035_b9_s2024',
    'attnbias_dstart_ml50_do035_b9_s7',
]
multitask_baseline_runs = [
    'multitask_refine_ml50_do035_s42',
    'multitask_refine_ml50_do035_s2024',
    'multitask_refine_ml50_do035_s7',
]

checks = [
    ('baseline', BASELINE_NDCG10_OUTPUT_DIR, baseline_runs),
    ('single-task attention bias', SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR, single_task_attnbias_runs),
    ('multi-task baseline', MULTITASK_BASELINE_OUTPUT_DIR, multitask_baseline_runs),
]

print('=' * 80)
for label, output_dir, run_names in checks:
    print(label)
    base = Path(output_dir)
    for run_name in run_names:
        run_dir = base / run_name
        print(' ', run_name, 'EXISTS' if run_dir.exists() else 'MISSING')
    print('-' * 80)


baseline
  refine_ml50_do035_s42 EXISTS
  refine_ml50_do035_s2024 EXISTS
  refine_ml50_do035_s7 EXISTS
--------------------------------------------------------------------------------
single-task attention bias
  attnbias_dstart_ml50_do035_b9_s42 EXISTS
  attnbias_dstart_ml50_do035_b9_s2024 EXISTS
  attnbias_dstart_ml50_do035_b9_s7 EXISTS
--------------------------------------------------------------------------------
multi-task baseline
  multitask_refine_ml50_do035_s42 EXISTS
  multitask_refine_ml50_do035_s2024 EXISTS
  multitask_refine_ml50_do035_s7 EXISTS
--------------------------------------------------------------------------------


## Check planned refine attention-bias multi-task runs


In [12]:
planned_attnbias_multitask_runs = [
    'multitask_attnbias_dstart_ml50_do035_b9_s42',
    'multitask_attnbias_dstart_ml50_do035_b9_s2024',
    'multitask_attnbias_dstart_ml50_do035_b9_s7',
    'multitask_attnbias_dstart_ml50_do035_b9_w01_s42',
    'multitask_attnbias_dstart_ml50_do035_b9_w01_s2024',
    'multitask_attnbias_dstart_ml50_do035_b9_w01_s7',
]

checks = [
    ('w1.0', MULTITASK_REFINE_ATTNBIAS_OUTPUT_DIR),
    ('w0.1', MULTITASK_REFINE_ATTNBIAS_W01_OUTPUT_DIR),
]

for label, output_dir in checks:
    base = Path(output_dir)
    print('=' * 80)
    print(f'Stage 3 refine attention-bias multi-task runs ({label})')
    for run_name in planned_attnbias_multitask_runs:
        if (label == 'w1.0' and '_w01_' in run_name) or (label == 'w0.1' and '_w01_' not in run_name):
            continue
        run_dir = base / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'OK')


Stage 3 refine attention-bias multi-task runs (w1.0)
multitask_attnbias_dstart_ml50_do035_b9_s42 EXISTS
multitask_attnbias_dstart_ml50_do035_b9_s2024 EXISTS
multitask_attnbias_dstart_ml50_do035_b9_s7 EXISTS
Stage 3 refine attention-bias multi-task runs (w0.1)
multitask_attnbias_dstart_ml50_do035_b9_w01_s42 EXISTS
multitask_attnbias_dstart_ml50_do035_b9_w01_s2024 EXISTS
multitask_attnbias_dstart_ml50_do035_b9_w01_s7 OK


## Train refine attention-bias multi-task w1.0 runs


In [ ]:
!python src/train_sasrec.py \
  --run_name multitask_attnbias_dstart_ml50_do035_b9_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_REFINE_ATTNBIAS_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_refine_attention_bias_multitask_ndcg10_v2/multitask_attnbias_dstart_ml50_do035_b9_s42
epoch=1, loss=3.0479
epoch=2, loss=1.7479
epoch=3, loss=1.5749
epoch=4, loss=1.4773
epoch=5, loss=1.4232
valid [task], Top5Acc: 0.4741, Top10Acc: 0.7317, Acc: 0.0583, MacroF1: 0.0782, TimeMAE: 69475.7882, TimeRMSE: 277601.1101, TimeMedAE: 860.2867
valid [full], NDCG@5: 0.5947, HR@5: 0.6861, NDCG@10: 0.6972, HR@10: 0.9948, MRR: 0.6097
valid [sampled], NDCG@5: 0.5151, HR@5: 0.5159, NDCG@10: 0.5188, HR@10: 0.5278, MRR: 0.5309
test [task], Top5Acc: 0.0536, Top10Acc: 0.4662, Acc: 0.0118, MacroF

In [ ]:
!python src/train_sasrec.py \
  --run_name multitask_attnbias_dstart_ml50_do035_b9_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_REFINE_ATTNBIAS_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_refine_attention_bias_multitask_ndcg10_v2/multitask_attnbias_dstart_ml50_do035_b9_s2024
epoch=1, loss=2.8451
epoch=2, loss=1.7460
epoch=3, loss=1.5668
epoch=4, loss=1.4792
epoch=5, loss=1.4259
valid [task], Top5Acc: 0.3531, Top10Acc: 0.6957, Acc: 0.0551, MacroF1: 0.0869, TimeMAE: 74203.5075, TimeRMSE: 290869.9884, TimeMedAE: 814.5342
valid [full], NDCG@5: 0.5915, HR@5: 0.6823, NDCG@10: 0.6608, HR@10: 0.8996, MRR: 0.5998
valid [sampled], NDCG@5: 0.5190, HR@5: 0.5192, NDCG@10: 0.5213, HR@10: 0.5266, MRR: 0.5326
test [task], Top5Acc: 0.2959, Top10Acc: 0.6225, Acc: 0.0149, Macr

In [ ]:
!python src/train_sasrec.py \
  --run_name multitask_attnbias_dstart_ml50_do035_b9_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_REFINE_ATTNBIAS_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_refine_attention_bias_multitask_ndcg10_v2/multitask_attnbias_dstart_ml50_do035_b9_s7
epoch=1, loss=3.2969
epoch=2, loss=1.7796
epoch=3, loss=1.6205
epoch=4, loss=1.5294
epoch=5, loss=1.4809
valid [task], Top5Acc: 0.2969, Top10Acc: 0.6944, Acc: 0.0502, MacroF1: 0.0859, TimeMAE: 71546.4460, TimeRMSE: 290715.7656, TimeMedAE: 2032.2478
valid [full], NDCG@5: 0.4900, HR@5: 0.6359, NDCG@10: 0.6047, HR@10: 0.9778, MRR: 0.4927
valid [sampled], NDCG@5: 0.3358, HR@5: 0.3615, NDCG@10: 0.3722, HR@10: 0.4751, MRR: 0.3582
test [task], Top5Acc: 0.1844, Top10Acc: 0.6760, Acc: 0.0410, MacroF

## Train refine attention-bias multi-task w0.1 runs


In [ ]:
!python src/train_sasrec.py \
  --run_name multitask_attnbias_dstart_ml50_do035_b9_w01_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 0.1 \
  --output_dir "$MULTITASK_REFINE_ATTNBIAS_W01_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_refine_attention_bias_multitask_w01_ndcg10_v2/multitask_attnbias_dstart_ml50_do035_b9_w01_s42
epoch=1, loss=0.8965
epoch=2, loss=0.4229
epoch=3, loss=0.3437
epoch=4, loss=0.3051
epoch=5, loss=0.2809
valid [task], Top5Acc: 0.4872, Top10Acc: 0.7646, Acc: 0.0603, MacroF1: 0.0930, TimeMAE: 73939.8953, TimeRMSE: 279427.1200, TimeMedAE: 831.4806
valid [full], NDCG@5: 0.6551, HR@5: 0.7716, NDCG@10: 0.7263, HR@10: 0.9967, MRR: 0.6467
valid [sampled], NDCG@5: 0.5590, HR@5: 0.5621, NDCG@10: 0.5659, HR@10: 0.5840, MRR: 0.5738
test [task], Top5Acc: 0.2827, Top10Acc: 0.4386, Acc: 0.0118

In [ ]:
!python src/train_sasrec.py \
  --run_name multitask_attnbias_dstart_ml50_do035_b9_w01_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 0.1 \
  --output_dir "$MULTITASK_REFINE_ATTNBIAS_W01_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_refine_attention_bias_multitask_w01_ndcg10_v2/multitask_attnbias_dstart_ml50_do035_b9_w01_s2024
epoch=1, loss=0.8832
epoch=2, loss=0.4327
epoch=3, loss=0.3458
epoch=4, loss=0.3034
epoch=5, loss=0.2811
valid [task], Top5Acc: 0.4981, Top10Acc: 0.7141, Acc: 0.0922, MacroF1: 0.1377, TimeMAE: 100511.0429, TimeRMSE: 326375.8715, TimeMedAE: 815.7832
valid [full], NDCG@5: 0.6555, HR@5: 0.7703, NDCG@10: 0.7214, HR@10: 0.9769, MRR: 0.6475
valid [sampled], NDCG@5: 0.5605, HR@5: 0.5653, NDCG@10: 0.5682, HR@10: 0.5892, MRR: 0.5745
test [task], Top5Acc: 0.2752, Top10Acc: 0.6384, Acc: 0.0

In [13]:
!python src/train_sasrec.py \
  --run_name multitask_attnbias_dstart_ml50_do035_b9_w01_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 0.1 \
  --output_dir "$MULTITASK_REFINE_ATTNBIAS_W01_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_refine_attention_bias_multitask_w01_ndcg10_v2/multitask_attnbias_dstart_ml50_do035_b9_w01_s7
epoch=1, loss=0.9030
epoch=2, loss=0.4477
epoch=3, loss=0.3571
epoch=4, loss=0.3177
epoch=5, loss=0.2935
valid [task], Top5Acc: 0.4659, Top10Acc: 0.7888, Acc: 0.0916, MacroF1: 0.1731, TimeMAE: 77743.6406, TimeRMSE: 293442.9292, TimeMedAE: 3468.6527
valid [full], NDCG@5: 0.6510, HR@5: 0.7863, NDCG@10: 0.7118, HR@10: 0.9721, MRR: 0.6358
valid [sampled], NDCG@5: 0.5347, HR@5: 0.5373, NDCG@10: 0.5427, HR@10: 0.5624, MRR: 0.5515
test [task], Top5Acc: 0.0754, Top10Acc: 0.5324, Acc: 0.0431

## Load run summaries


In [14]:
import json
from pathlib import Path
import pandas as pd

def rebuild_df(output_dir):
    rows = []
    for run_dir in sorted(Path(output_dir).iterdir()):
        if not run_dir.is_dir():
            continue
        summary_path = run_dir / 'metrics_summary.json'
        config_path = run_dir / 'config.json'
        if not summary_path.exists() or not config_path.exists():
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        config = json.loads(config_path.read_text(encoding='utf-8'))
        row = {
            'run_name': summary.get('run_name'),
            'run_dir': str(run_dir),
            'completed_at': summary.get('completed_at'),
            'best_epoch': summary.get('best_epoch'),
            'checkpoint_best': summary.get('checkpoint_best'),
            'checkpoint_last': summary.get('checkpoint_last'),
            'metrics_history': summary.get('metrics_history'),
            'config_path': str(config_path),
            'metrics_summary': str(summary_path),
            'maxlen': config.get('maxlen'),
            'dropout_rate': config.get('dropout_rate'),
            'hidden_units': config.get('hidden_units'),
            'seed': config.get('seed'),
            'selection_metric': config.get('selection_metric'),
            'use_time_attention_bias': config.get('use_time_attention_bias', False),
            'enable_time_prediction': config.get('enable_time_prediction', False),
            'time_delta_column': config.get('time_delta_column'),
            'time_prediction_target': config.get('time_prediction_target'),
            'time_loss_weight': config.get('time_loss_weight'),
            'time_target_transform': config.get('time_target_transform'),
            'time_modeling_mode': config.get('time_modeling_mode'),
        }
        for group_name in ['best_valid', 'best_test_at_best_valid', 'last_valid', 'last_test']:
            metrics = summary.get(group_name, {})
            for metric_name, value in metrics.items():
                row[f'{group_name}_{metric_name}'] = value
        rows.append(row)
    return pd.DataFrame(rows)


## Compare refine single-task / plain multi-task / attention-bias multi-task


In [24]:
baseline_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]
single_task_attnbias_runs = [
    'attnbias_dstart_ml50_do035_b9_s42',
    'attnbias_dstart_ml50_do035_b9_s2024',
    'attnbias_dstart_ml50_do035_b9_s7',
]
multitask_baseline_runs = [
    'multitask_refine_ml50_do035_s42',
    'multitask_refine_ml50_do035_s2024',
    'multitask_refine_ml50_do035_s7',
]
multitask_attnbias_runs = [
    'multitask_attnbias_dstart_ml50_do035_b9_s42',
    'multitask_attnbias_dstart_ml50_do035_b9_s2024',
    'multitask_attnbias_dstart_ml50_do035_b9_s7',
]
multitask_attnbias_w01_runs = [
    'multitask_attnbias_dstart_ml50_do035_b9_w01_s42',
    'multitask_attnbias_dstart_ml50_do035_b9_w01_s2024',
    'multitask_attnbias_dstart_ml50_do035_b9_w01_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG10_OUTPUT_DIR)
single_task_attnbias_df = rebuild_df(SINGLE_TASK_ATTNBIAS_NDCG10_OUTPUT_DIR)
multitask_baseline_df = rebuild_df(MULTITASK_BASELINE_OUTPUT_DIR)
multitask_attnbias_df = rebuild_df(MULTITASK_REFINE_ATTNBIAS_OUTPUT_DIR)
multitask_attnbias_w01_df = rebuild_df(MULTITASK_REFINE_ATTNBIAS_W01_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['variant'] = 'refine_single_task'

single_task_attnbias_subset = single_task_attnbias_df[single_task_attnbias_df['run_name'].isin(single_task_attnbias_runs)].copy()
single_task_attnbias_subset['variant'] = 'refine_attnbias_single_task'

multitask_baseline_subset = multitask_baseline_df[multitask_baseline_df['run_name'].isin(multitask_baseline_runs)].copy()
multitask_baseline_subset['variant'] = 'refine_multi_task_w1.0'

multitask_attnbias_subset = multitask_attnbias_df[multitask_attnbias_df['run_name'].isin(multitask_attnbias_runs)].copy()
multitask_attnbias_subset['variant'] = 'refine_attnbias_multi_task_w1.0'

multitask_attnbias_w01_subset = multitask_attnbias_w01_df[multitask_attnbias_w01_df['run_name'].isin(multitask_attnbias_w01_runs)].copy()
multitask_attnbias_w01_subset['variant'] = 'refine_attnbias_multi_task_w0.1'

df_compare = pd.concat(
    [
        baseline_subset,
        single_task_attnbias_subset,
        multitask_baseline_subset,
        multitask_attnbias_subset,
        multitask_attnbias_w01_subset,
    ],
    ignore_index=True,
)
df_compare = df_compare.sort_values(['variant', 'seed', 'run_name']).reset_index(drop=True)

# nested dict metric columns -> flat columns
nested_metric_cols = [
    'best_valid_full',
    'best_valid_sampled',
    'best_test_at_best_valid_full',
    'best_test_at_best_valid_sampled',
    'best_valid_task',
    'best_test_at_best_valid_task',
    'last_valid_full',
    'last_valid_sampled',
    'last_test_full',
    'last_test_sampled',
    'last_valid_task',
    'last_test_task',
]

for col in nested_metric_cols:
    if col in df_compare.columns:
        expanded = pd.json_normalize(df_compare[col]).add_prefix(col + '_')
        df_compare = pd.concat([df_compare.drop(columns=[col]), expanded], axis=1)

id_cols = [
    'run_name',
    'seed',
    'variant',
    'maxlen',
    'dropout_rate',
    'selection_metric',
    'time_loss_weight',
    'best_epoch',
]

metric_prefixes = (
    'best_valid_',
    'best_test_at_best_valid_',
    'last_valid_',
    'last_test_',
)

metric_cols = [c for c in df_compare.columns if c.startswith(metric_prefixes)]

def metric_sort_key(col):
    if col.startswith('best_valid_'):
        group_rank = 0
        name = col[len('best_valid_'):]
    elif col.startswith('best_test_at_best_valid_'):
        group_rank = 1
        name = col[len('best_test_at_best_valid_'):]
    elif col.startswith('last_valid_'):
        group_rank = 2
        name = col[len('last_valid_'):]
    elif col.startswith('last_test_'):
        group_rank = 3
        name = col[len('last_test_'):]
    else:
        group_rank = 99
        name = col

    metric_order = [
        'full_ndcg@10', 'full_hr@10', 'full_ndcg@5', 'full_hr@5', 'full_mrr',
        'sampled_ndcg@10', 'sampled_hr@10', 'sampled_ndcg@5', 'sampled_hr@5', 'sampled_mrr',
        'task_accuracy', 'task_macro_f1', 'task_top5_accuracy', 'task_top10_accuracy',
        'task_time_mae', 'task_time_rmse', 'task_time_median_ae',
    ]

    try:
        metric_rank = metric_order.index(name)
    except ValueError:
        metric_rank = 999

    return (group_rank, metric_rank, name)

metric_cols = sorted(metric_cols, key=metric_sort_key)

display_cols = [c for c in id_cols if c in df_compare.columns] + metric_cols

import pandas as pd

print('num display cols =', len(display_cols))
print(display_cols)

with pd.option_context(
    'display.max_columns', None,
    'display.max_colwidth', None,
    'display.width', 1000,
):
    display(df_compare[display_cols])



num display cols = 104
['run_name', 'seed', 'variant', 'maxlen', 'dropout_rate', 'selection_metric', 'time_loss_weight', 'best_epoch', 'best_valid_full_ndcg@10', 'best_valid_full_hr@10', 'best_valid_full_ndcg@5', 'best_valid_full_hr@5', 'best_valid_full_mrr', 'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10', 'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5', 'best_valid_sampled_mrr', 'best_valid_task_accuracy', 'best_valid_task_macro_f1', 'best_valid_task_top5_accuracy', 'best_valid_task_top10_accuracy', 'best_valid_task_time_mae', 'best_valid_task_time_rmse', 'best_valid_task_time_median_ae', 'best_valid_full_mean_rank', 'best_valid_full_median_rank', 'best_valid_full_num_eval_users', 'best_valid_sampled_mean_rank', 'best_valid_sampled_median_rank', 'best_valid_sampled_num_eval_users', 'best_valid_task_top1_accuracy', 'best_test_at_best_valid_full_ndcg@10', 'best_test_at_best_valid_full_hr@10', 'best_test_at_best_valid_full_ndcg@5', 'best_test_at_best_valid_full_hr@5', 'b

,run_name,seed,variant,maxlen,dropout_rate,selection_metric,time_loss_weight,best_epoch,best_valid_full_ndcg@10,best_valid_full_hr@10,best_valid_full_ndcg@5,best_valid_full_hr@5,best_valid_full_mrr,best_valid_sampled_ndcg@10,best_valid_sampled_hr@10,best_valid_sampled_ndcg@5,best_valid_sampled_hr@5,best_valid_sampled_mrr,best_valid_task_accuracy,best_valid_task_macro_f1,best_valid_task_top5_accuracy,best_valid_task_top10_accuracy,best_valid_task_time_mae,best_valid_task_time_rmse,best_valid_task_time_median_ae,best_valid_full_mean_rank,best_valid_full_median_rank,best_valid_full_num_eval_users,best_valid_sampled_mean_rank,best_valid_sampled_median_rank,best_valid_sampled_num_eval_users,best_valid_task_top1_accuracy,best_test_at_best_valid_full_ndcg@10,best_test_at_best_valid_full_hr@10,best_test_at_best_valid_full_ndcg@5,best_test_at_best_valid_full_hr@5,best_test_at_best_valid_full_mrr,best_test_at_best_valid_sampled_ndcg@10,best_test_at_best_valid_sampled_hr@10,best_test_at_best_valid_sampled_ndcg@5,best_test_at_best_valid_sampled_hr@5,best_test_at_best_valid_sampled_mrr,best_test_at_best_valid_task_accuracy,best_test_at_best_valid_task_macro_f1,best_test_at_best_valid_task_top5_accuracy,best_test_at_best_valid_task_top10_accuracy,best_test_at_best_valid_task_time_mae,best_test_at_best_valid_task_time_rmse,best_test_at_best_valid_task_time_median_ae,best_test_at_best_valid_full_mean_rank,best_test_at_best_valid_full_median_rank,best_test_at_best_valid_full_num_eval_users,best_test_at_best_valid_sampled_mean_rank,best_test_at_best_valid_sampled_median_rank,best_test_at_best_valid_sampled_num_eval_users,best_test_at_best_valid_task_top1_accuracy,last_valid_full_ndcg@10,last_valid_full_hr@10,last_valid_full_ndcg@5,last_valid_full_hr@5,last_valid_full_mrr,last_valid_sampled_ndcg@10,last_valid_sampled_hr@10,last_valid_sampled_ndcg@5,last_valid_sampled_hr@5,last_valid_sampled_mrr,last_valid_task_accuracy,last_valid_task_macro_f1,last_valid_task_top5_accuracy,last_valid_task_top10_accuracy,last_valid_task_time_mae,last_valid_task_time_rmse,last_valid_task_time_median_ae,last_valid_full_mean_rank,last_valid_full_median_rank,last_valid_full_num_eval_users,last_valid_sampled_mean_rank,last_valid_sampled_median_rank,last_valid_sampled_num_eval_users,last_valid_task_top1_accuracy,last_test_full_ndcg@10,last_test_full_hr@10,last_test_full_ndcg@5,last_test_full_hr@5,last_test_full_mrr,last_test_sampled_ndcg@10,last_test_sampled_hr@10,last_test_sampled_ndcg@5,last_test_sampled_hr@5,last_test_sampled_mrr,last_test_task_accuracy,last_test_task_macro_f1,last_test_task_top5_accuracy,last_test_task_top10_accuracy,last_test_task_time_mae,last_test_task_time_rmse,last_test_task_time_median_ae,last_test_full_mean_rank,last_test_full_median_rank,last_test_full_num_eval_users,last_test_sampled_mean_rank,last_test_sampled_median_rank,last_test_sampled_num_eval_users,last_test_task_top1_accuracy
0,multitask_attnbias_dstart_ml50_do035_b9_w01_s7,7,refine_attnbias_multi_task_w0.1,50,0.35,full_valid_ndcg@10,0.1,50,0.747438,0.971785,0.718038,0.875882,0.678257,0.556673,0.619371,0.531293,0.539745,0.555830,0.059414,0.070785,0.400705,0.725583,84283.170385,290334.467376,6782.013222,2.651926,1.0,7372,15.498101,1.0,7372,0.059414,0.910145,1.000000,0.909194,0.997148,0.878992,0.406329,0.628005,0.327147,0.380687,0.365058,0.122776,0.048669,0.538232,0.714654,14519.626672,66724.220192,163.442845,1.292272,1.0,7363,9.401603,8.0,7363,0.122776,0.747438,0.971785,0.718038,0.875882,0.678257,0.556673,0.619371,0.531293,0.539745,0.555830,0.059414,0.070785,0.400705,0.725583,84283.170385,290334.467376,6782.013222,2.651926,1.0,7372,15.498101,1.0,7372,0.059414,0.910145,1.000000,0.909194,0.997148,0.878992,0.406329,0.628005,0.327147,0.380687,0.365058,0.122776,0.048669,0.538232,0.714654,14519.626672,66724.220192,163.442845,1.292272,1.0,7363,9.401603,8.0,7363,0.122776
1,multitask_attnbias_dstart_ml50_do035_b9_w01_s42,42,refine_attnbias_multi_task_w0.1,50,0.35,full_valid_ndcg@10,0.1,25,0.

In [25]:
import pandas as pd

candidate_cols = [
    c for c in df_compare.columns
    if c.startswith(('best_valid_', 'best_test_at_best_valid_'))
]

summary_metric_cols = []
for c in candidate_cols:
    converted = pd.to_numeric(df_compare[c], errors='coerce')
    if converted.notna().any():
        df_compare[c] = converted
        summary_metric_cols.append(c)

def summary_metric_sort_key(col):
    if col.startswith('best_valid_'):
        group_rank = 0
        name = col[len('best_valid_'):]
    elif col.startswith('best_test_at_best_valid_'):
        group_rank = 1
        name = col[len('best_test_at_best_valid_'):]
    else:
        group_rank = 99
        name = col

    metric_order = [
        'full_ndcg@10', 'full_hr@10', 'full_ndcg@5', 'full_hr@5', 'full_mrr',
        'sampled_ndcg@10', 'sampled_hr@10', 'sampled_ndcg@5', 'sampled_hr@5', 'sampled_mrr',
        'task_accuracy', 'task_macro_f1', 'task_top5_accuracy', 'task_top10_accuracy',
        'task_time_mae', 'task_time_rmse', 'task_time_median_ae',
    ]

    try:
        metric_rank = metric_order.index(name)
    except ValueError:
        metric_rank = 999

    return (group_rank, metric_rank, name)

summary_metric_cols = sorted(summary_metric_cols, key=summary_metric_sort_key)

print('num summary metric cols =', len(summary_metric_cols))
print(summary_metric_cols)

summary_compare = df_compare.groupby('variant')[summary_metric_cols].agg(['mean', 'std'])

with pd.option_context(
    'display.max_columns', None,
    'display.max_colwidth', None,
    'display.width', 2000,
):
    display(summary_compare)


num summary metric cols = 48
['best_valid_full_ndcg@10', 'best_valid_full_hr@10', 'best_valid_full_ndcg@5', 'best_valid_full_hr@5', 'best_valid_full_mrr', 'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10', 'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5', 'best_valid_sampled_mrr', 'best_valid_task_accuracy', 'best_valid_task_macro_f1', 'best_valid_task_top5_accuracy', 'best_valid_task_top10_accuracy', 'best_valid_task_time_mae', 'best_valid_task_time_rmse', 'best_valid_task_time_median_ae', 'best_valid_full_mean_rank', 'best_valid_full_median_rank', 'best_valid_full_num_eval_users', 'best_valid_sampled_mean_rank', 'best_valid_sampled_median_rank', 'best_valid_sampled_num_eval_users', 'best_valid_task_top1_accuracy', 'best_test_at_best_valid_full_ndcg@10', 'best_test_at_best_valid_full_hr@10', 'best_test_at_best_valid_full_ndcg@5', 'best_test_at_best_valid_full_hr@5', 'best_test_at_best_valid_full_mrr', 'best_test_at_best_valid_sampled_ndcg@10', 'best_test_at_best_valid_sam

best_valid_full_ndcg@10           best_valid_full_hr@10           best_valid_full_ndcg@5           best_valid_full_hr@5           best_valid_full_mrr           best_valid_sampled_ndcg@10           best_valid_sampled_hr@10           best_valid_sampled_ndcg@5           best_valid_sampled_hr@5           best_valid_sampled_mrr           best_valid_task_accuracy           best_valid_task_macro_f1           best_valid_task_top5_accuracy           best_valid_task_top10_accuracy           best_valid_task_time_mae              best_valid_task_time_rmse              best_valid_task_time_median_ae              best_valid_full_mean_rank           best_valid_full_median_rank      best_valid_full_num_eval_users            best_valid_sampled_mean_rank           best_valid_sampled_median_rank      best_valid_sampled_num_eval_users            best_valid_task_top1_accuracy           best_test_at_best_valid_full_ndcg@10           best_test_at_best_valid_full_hr@10           best_test_at_best_valid_full_ndcg@5           best_test_at_best_valid_full_hr@5           best_test_at_best_valid_full_mrr           best_test_at_best_valid_sampled_ndcg@10           best_test_at_best_valid_sampled_hr@10           best_test_at_best_valid_sampled_ndcg@5           best_test_at_best_valid_sampled_hr@5           best_test_at_best_valid_sampled_mrr           best_test_at_best_valid_task_accuracy           best_test_at_best_valid_task_macro_f1           best_test_at_best_valid_task_top5_accuracy           best_test_at_best_valid_task_top10_accuracy           best_test_at_best_valid_task_time_mae              best_test_at_best_valid_task_time_rmse              best_test_at_best_valid_task_time_median_ae             best_test_at_best_valid_full_mean_rank           best_test_at_best_valid_full_median_rank          best_test_at_best_valid_full_num_eval_users            best_test_at_best_valid_sampled_mean_rank           best_test_at_best_valid_sampled_median_rank            \
                                                   mean       std                  mean       std                   mean       std                 mean       std                mean       std                       mean       std                     mean       std                      mean       std                    mean       std                   mean       std                     mean       std                     mean       std                          mean       std                           mean       std                     mean          std                      mean          std                           mean          std                      mean       std                        mean  std                           mean        std                         mean       std                           mean  std                              mean        std                          mean       std                                 mean       std                               mean       std                                mean       std                              mean       std                             mean       std                                    mean       std                                  mean       std                                   mean       std                                 mean       std                                mean       std                                  mean       std                                  mean       std                                       mean       std                                        mean       std                                  mean          std                                   mean          std                                        mean         std                                   mean       std                                     mean      std                                        mean        std                                      mean       std                                        mean       std   
variant                         

## What to look at

Read the results in this order:
- compare `refine_single_task` vs `refine_attnbias_single_task`
- compare `refine_multi_task_w1.0` vs `refine_attnbias_multi_task_w1.0`
- compare `refine_attnbias_multi_task_w1.0` vs `refine_attnbias_multi_task_w0.1`
- then decide whether refine behaves like anchor or shows a different trade-off pattern
